pip install git+https://github.com/openai/CLIP.git 
Tải lệnh này

Cell 1

In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 98.2 MB/s eta 0:00:00:00:0100:01


In [3]:
# Dữ liệu 
import pandas as pd
import numpy as np

# Ảnh & PyTorch 
import torch
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Tìm kiếm tương đồng
import faiss
from sklearn.metrics.pairwise import cosine_similarity

# Vẽ biểu đồ 
import matplotlib.pyplot as plt
import seaborn as sns

# Tiện ích
import os
from tqdm import tqdm

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 20)      
plt.rcParams['figure.figsize'] = (10, 5)    
sns.set_style('whitegrid')                  


print('Import thư viện thành công!')
print(f'PyTorch version : {torch.__version__}')
# Kiểm tra máy có GPU không. GPU giúp chạy nhanh hơn CPU rất nhiều.
# Laptop bình thường thường sẽ hiện 'cpu' – không sao, vẫn chạy được.
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

Import thư viện thành công!
PyTorch version : 2.10.0+cu128
Device: cuda


Cell 2

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os

# ĐƯỜNG DẪN ĐÚNG DỰA TRÊN ẢNH GOOGLE DRIVE CỦA BẠN:
DATA_DIR = '/content/drive/MyDrive/DoAnPython/DuLieuPython'

CSV_PATH = os.path.join(DATA_DIR, 'train.csv')
# Vì 'train_images.zip' đang là file nén, bạn cứ khai báo đường dẫn tệp trước:
IMAGE_ZIP_PATH = os.path.join(DATA_DIR, 'train_images.zip') 

PROCESSED = '../data/processed/'
RESULTS = '../results/'
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

# Kiểm tra lại đường dẫn
for p in [CSV_PATH, IMAGE_ZIP_PATH]:
    status = 'Đã tìm thấy tệp/thư mục' if os.path.exists(p) else 'Không tìm thấy – kiểm tra lại đường dẫn!'
    print(f'{status}  {p}')

Đã tìm thấy tệp/thư mục  /content/drive/MyDrive/DoAnPython/DuLieuPython/train.csv
Đã tìm thấy tệp/thư mục  /content/drive/MyDrive/DoAnPython/DuLieuPython/train_images.zip


Cell 3

In [7]:
class ShopeeDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['image']
        img_path = os.path.join(self.img_dir, img_name)
        image = self.transform(Image.open(img_path).convert("RGB"))
        text = str(self.df.iloc[idx]['title'])
        text_token = clip.tokenize([text], truncate=True).squeeze(0)
        return image, text_token

In [8]:
df = pd.read_csv(CSV_PATH)

print(f'Train: {df.shape[0]:,} dòng x {df.shape[1]:,} cột')

df.head()

Train: 34,250 dòng x 5 cột


,posting_id,image,image_phash,title,label_group
0,train_129225211,0000a68812bc7e98c42888dfb1c07da0.jpg,94974f937d4c2433,Paper Bag Victoria Secret,249114794
1,train_3386243561,00039780dfc94d01db8676fe789ecd05.jpg,af3f9460c2838f0f,"Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DO...",2937985045
2,train_2288590299,000a190fdd715a2a36faed16e2c65df7.jpg,b94cb00ed3e50f78,Maling TTS Canned Pork Luncheon Meat 397 gr,2395904891
3,train_2406599165,00117e4fc239b1b641ff08340b429633.jpg,8514fc58eafea283,Daster Batik Lengan pendek - Motif Acak / Camp...,4093212188
4,train_3369186413,00136d1cf4edede0203f32f05f660588.jpg,a6f319f924ad708c,Nescafe \xc3\x89clair Latte 220ml,3648931069


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34250 entries, 0 to 34249
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   posting_id   34250 non-null  object
 1   image        34250 non-null  object
 2   image_phash  34250 non-null  object
 3   title        34250 non-null  object
 4   label_group  34250 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [10]:
candidate_df = pd.read_csv(CSV_PATH)

Cell 4

In [11]:
import torch
import clip
from PIL import Image
import torch.nn.functional as F
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

ModuleNotFoundError: No module named 'clip'

Cell 5 trích xuất vecto

In [ ]:
import pandas as pd
from PIL import Image
import torch
import clip
import os
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch_xla.core.xla_model as xm

csv_path = '/content/drive/MyDrive/DoAnPython/project/data/raw/train.csv'
image_folder = '/content/drive/MyDrive/DoAnPython/project/data/raw/train_images'

candidate_df = pd.read_csv(csv_path)

device = xm.xla_device()
model, preprocess = clip.load("ViT-B/32", device="cpu")
model = model.to(device)

class ShopeeDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['image']
        img_path = os.path.join(self.img_dir, img_name)
        image = self.transform(Image.open(img_path).convert("RGB"))
        text = str(self.df.iloc[idx]['title'])
        text_token = clip.tokenize([text], truncate=True).squeeze(0)
        return image, text_token

dataset = ShopeeDataset(candidate_df, image_folder, preprocess)
dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=4)

image_features_list = []
text_features_list = []

with torch.inference_mode():
    for images, texts in tqdm(dataloader):
        images = images.to(device)
        texts = texts.to(device)
        
        img_features = model.encode_image(images)
        txt_features = model.encode_text(texts)
        
        image_features_list.append(img_features.cpu())
        text_features_list.append(txt_features.cpu())
        xm.mark_step()

image_features = torch.cat(image_features_list)
text_features = torch.cat(text_features_list)

print(image_features.shape)
print(text_features.shape)

100%|██████████| 1071/1071 [1:32:15<00:00,  5.17s/it] 

torch.Size([34250, 512])
torch.Size([34250, 512])


Cell 6 tính toán ma trận tương đồng

In [26]:
import torch.nn.functional as F

# 1. Chuẩn hóa L2 (L2 Normalization) cho cả 2 tập vector
# Việc này đưa tất cả các vector về cùng độ dài bằng 1, giúp phép nhân ma trận phía sau tương đương với việc tính Cosine Similarity.
image_features_norm = F.normalize(image_features, p=2, dim=1)
text_features_norm = F.normalize(text_features, p=2, dim=1)

# 2. Nhân ma trận (Matrix Multiplication)
# Nhân ma trận vector văn bản với ma trận chuyển vị (Transpose - ký hiệu .T) của vector hình ảnh.
# Kết quả sẽ ra một ma trận chứa điểm số tương đồng của mọi cặp text-image.
cosine_similarity_matrix = torch.mm(text_features_norm, image_features_norm.T)

# In thử kích thước ma trận để kiểm tra
print("Kích thước ma trận tương đồng:", cosine_similarity_matrix.shape)

Kích thước ma trận tương đồng: torch.Size([34250, 34250])


Cell 7 tính toán metric

In [11]:
print(candidate_df.columns)

Index(['posting_id', 'image', 'image_phash', 'title', 'label_group'], dtype='str')


In [42]:
import numpy as np
import pandas as pd
import torch
import os

group_to_indices = candidate_df.groupby('label_group').indices
labels = candidate_df['label_group'].values
num_samples = len(labels)

ap_scores = []
p1, r1 = [], []
p5, r5 = [], []
p10, r10 = [], []

BATCH_SIZE = 1024

for start_idx in range(0, num_samples, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, num_samples)
    sim_batch = cosine_similarity_matrix[start_idx:end_idx]
    sorted_batch_indices = torch.argsort(sim_batch, descending=True, dim=-1).cpu().numpy()
    
    for i in range(start_idx, end_idx):
        retrieved_indices = sorted_batch_indices[i - start_idx]
        query_label = labels[i]
        ground_truth_indices = group_to_indices[query_label]
        
        is_relevant = np.isin(retrieved_indices, ground_truth_indices)
        if is_relevant.any():
            hit_positions = np.where(is_relevant)[0] + 1
            hits_cum = np.arange(1, len(hit_positions) + 1)
            ap = np.sum(hits_cum / hit_positions) / len(ground_truth_indices)
        else:
            ap = 0.0
        ap_scores.append(ap)
        
        gt_len = len(ground_truth_indices)
        
        h1 = np.isin(retrieved_indices[:1], ground_truth_indices).sum()
        p1.append(h1 / 1)
        r1.append(h1 / gt_len if gt_len > 0 else 0.0)
        
        h5 = np.isin(retrieved_indices[:5], ground_truth_indices).sum()
        p5.append(h5 / 5)
        r5.append(h5 / gt_len if gt_len > 0 else 0.0)
        
        h10 = np.isin(retrieved_indices[:10], ground_truth_indices).sum()
        p10.append(h10 / 10)
        r10.append(h10 / gt_len if gt_len > 0 else 0.0)

candidate_df['full_ap'] = ap_scores

summary_metrics = pd.DataFrame({
    'Metric': ['mAP Toàn tập', 'Precision@1', 'Recall@1', 'Precision@5', 'Recall@5', 'Precision@10', 'Recall@10'],
    'Value': [np.mean(ap_scores), np.mean(p1), np.mean(r1), np.mean(p5), np.mean(r5), np.mean(p10), np.mean(r10)]
})

summary_metrics['Value'] = summary_metrics['Value'].round(4)

os.makedirs('../results', exist_ok=True)
summary_metrics.to_csv('../results/tuan3_hung_summary_metrics.csv', index=False)

summary_metrics

,Metric,Value
0,mAP Toàn tập,0.2049
1,Precision@1,0.2459
2,Recall@1,0.0685
3,Precision@5,0.1596
4,Recall@5,0.1913
5,Precision@10,0.1172
6,Recall@10,0.2566


Cell 8

In [ ]:
import torch
import numpy as np
import pandas as pd
from collections import Counter
import os
import matplotlib.pyplot as plt
from PIL import Image

image_folder = '/content/drive/MyDrive/DoAnPython/project/data/raw/train_images'
K = 5
topk_indices = torch.topk(cosine_similarity_matrix, k=K, dim=1)[1].cpu().numpy()
label_groups = candidate_df['label_group'].values
num_samples = len(label_groups)
wrong_counts = []
false_positives = []

for i in range(num_samples):
    query_label = label_groups[i]
    retrieved_indices = topk_indices[i]
    retrieved_labels = label_groups[retrieved_indices]
    wrong_in_topk = np.sum(retrieved_labels != query_label)
    wrong_counts.append(wrong_in_topk)
    for pred_idx in retrieved_indices:
        if label_groups[pred_idx] != query_label:
            false_positives.append(pred_idx)

candidate_df['wrong_count_top5'] = wrong_counts
most_wrong_queries = candidate_df.sort_values(by='wrong_count_top5', ascending=False).head(5)

print("--- TOP 5 QUERIES TRẢ VỀ SAI NHIỀU NHẤT VÀ HÌNH ẢNH MINH HỌA ---")
error_records = []

for idx, row in most_wrong_queries.iterrows():
    query_label = row['label_group']
    retrieved_indices = topk_indices[idx]
    print(f"\nQuery Index: {idx} | Title: '{row['title']}' | Label: {query_label} | Sai: {row['wrong_count_top5']}/{K}")
    
    fig, axes = plt.subplots(1, K, figsize=(15, 3))
    for rank, pred_idx in enumerate(retrieved_indices):
        pred_row = candidate_df.iloc[pred_idx]
        pred_label = label_groups[pred_idx]
        is_match = (pred_label == query_label)
        
        if not is_match:
            error_records.append({
                'query_index': idx,
                'query_title': row['title'],
                'query_label': query_label,
                'rank': rank + 1,
                'mismatch_image_index': pred_idx,
                'mismatch_image_file': pred_row['image'],
                'mismatch_image_title': pred_row['title'],
                'mismatch_image_label': pred_label
            })
        
        img_path = os.path.join(image_folder, pred_row['image'])
        if os.path.exists(img_path):
            img = Image.open(img_path)
            axes[rank].imshow(img)
        axes[rank].axis('off')
        color = 'green' if is_match else 'red'
        axes[rank].set_title(f"Top {rank+1}\n{'MATCH' if is_match else 'MISMATCH'}", color=color, fontsize=10)
    plt.tight_layout()
    plt.show()

fp_counts = Counter(false_positives)
most_confusing_images = fp_counts.most_common(5)

print("\n--- TOP 5 BỨC ẢNH GÂY NHIỄU / DỄ BỊ NHẬN NHẦM NHẤT ---")
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for rank, (img_idx, count) in enumerate(most_confusing_images):
    row = candidate_df.iloc[img_idx]
    print(f"Ảnh Index: {img_idx} | File: {row['image']} | Bị nhận nhầm: {count} lần")
    img_path = os.path.join(image_folder, row['image'])
    if os.path.exists(img_path):
        img = Image.open(img_path)
        axes[rank].imshow(img)
    axes[rank].axis('off')
    axes[rank].set_title(f"Confused: {count}\nIndex: {img_idx}", color='blue', fontsize=10)
plt.tight_layout()
plt.show()

os.makedirs('../results', exist_ok=True)
pd.DataFrame(error_records).to_csv('../results/tuan3_hung_error_analysis.csv', index=False)

NameError: name 'cosine_similarity_matrix' is not defined

# CELL 9: BIỆN LUẬN & PHÂN TÍCH LỖI (ERROR ANALYSIS)

Dựa trên kết quả thực nghiệm và trực quan hóa tại **Cell 8**, dưới đây là tổng hợp ngắn gọn các nguyên nhân cốt lõi khiến mô hình **CLIP (ViT-B/32)** dự đoán sai lệch giữa văn bản (Query) và hình ảnh:

---

### 1. Các dạng ảnh hệ thống dễ nhận nhầm nhất (False Positives)

* **Ảnh có phông nền trắng Studio (Clean Background):** Sản phẩm chụp biệt lập trên nền trắng tạo ra đặc trưng thị giác rất mạnh. Mô hình có xu hướng kéo các vector ảnh có cùng phong cách chụp lại gần nhau bất kể chúng thuộc ngành hàng nào.
* **Bao bì và khối hình học tương đồng:** Các vật thể dạng khối như *cuộn băng keo, hộp giấy vuông, tuýp mỹ phẩm...* chia sẻ chung các đặc trưng thô (coarse-grained). Do cơ chế cắt patch lớn của `ViT-B/32`, CLIP không đủ nhạy để nhận diện các chi tiết siêu nhỏ như nét chữ thương hiệu hay logo trên nhãn mác.

---

### 2. Nguyên nhân sai lệch từ phía Query văn bản

* **Nhiễu từ khóa TMĐT (Text Noise):** Tiêu đề sản phẩm Shopee chứa nhiều từ viết tắt, teencode tiếng Việt, thông số kỹ thuật viết liền hoặc ký tự quảng cáo (*"Freeship", "[Mã giảm]"*). Bộ mã hóa văn bản (Text Encoder) của CLIP gốc không hiểu được các cấu trúc phi tự nhiên này, dẫn đến sinh vector embedding sai lệch.
* **Nhiễu gán nhãn dữ liệu gốc (Label Noise):** Nhiều nhà bán hàng đăng tải sản phẩm giống hệt nhau về cả ảnh lẫn chữ nhưng hệ thống dữ liệu lại xếp vào các mã `label_group` khác nhau. Khi CLIP tìm kiếm ra chính xác ảnh giống hệt, thuật toán đối sánh nhãn cứng vẫn tính là **Sai (Mismatch)**, làm sụt giảm điểm metric tổng thể một cách oan uổng.